# 6.4 Deployment: FastAPI, Docker, Streaming

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langgraph==1.2.16"

## Your API key

Some cells in this notebook call a model and need your own OpenAI API key. Paste it when asked. It is stored only in this notebook session, never saved to the file.

In [ ]:
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

**Prerequisites:** 4.2 (StateGraph basics), 4.4 (streaming and interrupts), 5.1 (persistence and checkpointing). Every demo runs with no API key on a stub graph; the production variant notes where an `OPENAI_API_KEY` takes over.

## Learning objectives

- Wrap a compiled graph in a FastAPI app with a sync `/chat` endpoint and a streaming `/chat/stream` endpoint using Server-Sent Events.
- Stream tokens and graph events correctly: multi-mode `astream`, the `[DONE]` terminator, SSE headers, and handling clients that disconnect mid-stream.
- Replace in-memory checkpointing with a shared Postgres checkpointer so state survives restarts and works across replicas.
- Configure an agent service properly: secrets from the environment, health checks, and a Dockerfile plus docker-compose that actually runs it.
- Explain the production concurrency model: why you scale with replicas plus a shared checkpointer, not with workers sharing nothing.

## The gap between the notebook and the server

In every lesson so far, the graph lived in your process. You compiled it, invoked it, and when the process died, the state died with it. A deployed agent flips all three of those assumptions. Multiple users call it at once. The process restarts (deploys, crashes, the platform sleeping your free tier). The frontend wants tokens as they are generated, not a JSON blob after ninety seconds. Deployment is the lesson where each of those flips gets a concrete answer.

The shape of the answer is deliberately boring: a FastAPI app, a Postgres database, a Dockerfile. Boring is the point. Every production LangGraph deployment you will see in the wild is a variation on this shape. Learn the shape once and you can read any of them.

## The endpoint shape

Two endpoints cover nearly every agent product. `/chat` takes a message and a thread id and returns the final answer. `/chat/stream` takes the same and streams the run as Server-Sent Events. A third endpoint, `/health`, exists for the platform, not for users.

Start with the boring one. The key line is the `config` dict: the `thread_id` is what ties a request to the right conversation in the checkpointer, exactly as lesson 5.1 taught.

In [ ]:
# pip install fastapi==0.159.1 uvicorn==0.54.0 langgraph==1.2.16  (no API key needed)
from fastapi import FastAPI
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, MessagesState, START, END

def answer(state: MessagesState) -> dict:
    last = state["messages"][-1].content
    return {"messages": [{"role": "assistant", "content": f"echo: {last}"}]}

builder = StateGraph(MessagesState)
builder.add_node("answer", answer)
builder.add_edge(START, "answer")
builder.add_edge("answer", END)
graph = builder.compile()          # demo graph; checkpointer comes later in this lesson

app = FastAPI(title="agent-api")

class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=4000)
    thread_id: str = Field(min_length=1, max_length=128)

@app.get("/health")
def health() -> dict:
    return {"status": "ok"}

@app.post("/chat")
def chat(body: ChatRequest) -> dict:
    config = {"configurable": {"thread_id": body.thread_id}}
    result = graph.invoke({"messages": [("user", body.message)]}, config)
    return {"reply": result["messages"][-1].content, "thread_id": body.thread_id}

Run it with `uvicorn app:app --host 0.0.0.0 --port 8000` and post to it with curl:

In [ ]:
!curl -X POST localhost:8000/chat \
!  -H 'Content-Type: application/json' \
!  -d '{"message": "hello from Berlin", "thread_id": "user-42"}'
# {"reply":"echo: hello from Berlin","thread_id":"user-42"}

Three things to notice. First, the `thread_id` is user-supplied. In production you derive it from the authenticated user and the conversation, never from free-form client input, because anyone who can guess another user's thread id can read their conversation. Second, `/health` returns a constant. Load balancers and orchestrators poll it to decide whether your container is alive; it must be cheap and must not touch the model. Third, this is a sync endpoint calling `graph.invoke`. For real agents you will mostly serve the streaming endpoint instead, because nobody waits ninety seconds for a JSON response.

## Streaming to the frontend with SSE

A chat UI feels broken if it shows nothing until the whole answer is ready. The fix is Server-Sent Events: the server holds one HTTP response open and pushes `data:` lines down it as the run progresses. SSE is one-directional, works over plain HTTP/1.1, and browsers reconnect automatically with `EventSource`. Lesson 4.4 taught graph streaming; this is where it meets the network.

The correct stream call uses several modes at once. When you pass a list of `stream_mode` values, `astream` yields `(mode, chunk)` tuples: `"messages"` for LLM token chunks, `"updates"` for per-node completions, `"custom"` for anything your nodes wrote with `get_stream_writer()`. That tuple form is load-bearing; with a single mode you get raw chunks and no mode label.

In [ ]:
import json
from fastapi import Request
from fastapi.responses import StreamingResponse

@app.post("/chat/stream")
async def chat_stream(body: ChatRequest, request: Request):
    config = {"configurable": {"thread_id": body.thread_id}}

    async def gen():
        try:
            async for mode, chunk in graph.astream(
                {"messages": [("user", body.message)]},
                config,
                stream_mode=["messages", "updates"],
            ):
                if await request.is_disconnected():
                    break                                   # client went away; stop the run
                if mode == "messages":
                    msg_chunk, _meta = chunk                # tuple: (message chunk, metadata)
                    if msg_chunk.content:
                        yield f"data: {json.dumps({'type': 'token', 'content': msg_chunk.content})}\n\n"
                elif mode == "updates":
                    yield f"data: {json.dumps({'type': 'event', 'nodes': list(chunk)})}\n\n"
        except Exception as exc:                            # never let the stream die silently
            yield f"data: {json.dumps({'type': 'error', 'error': str(exc)})}\n\n"
        yield "data: [DONE]\n\n"                            # terminator the frontend watches for

    return StreamingResponse(
        gen(),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "Connection": "keep-alive",
            "X-Accel-Buffering": "no",                      # see below; this line matters
        },
    )

The `X-Accel-Buffering: no` header is the detail that breaks most first deployments. If your app sits behind nginx or Traefik (and in Docker it will), the proxy buffers the response before forwarding it. Your beautiful token stream arrives at the browser as one lump after the run finishes, and you spend a day debugging your generator before realising the proxy was the buffer. The header tells the proxy not to buffer; pair it with `proxy_buffering off` in the nginx config if you manage the proxy yourself.

The `is_disconnected` check is the detail that saves your API bill. A user who closes the tab mid-answer leaves the generator running on the server by default, and the run keeps calling the model for nobody. Checking each iteration and breaking out stops the spend. The `try/except` around the whole loop exists because an exception inside an async generator otherwise closes the connection with no explanation; the frontend deserves an `error` event it can render.

The frontend side is small. With `EventSource` or `fetch` plus a reader, it appends each `token` event to the message bubble and stops at `[DONE]`. You do not need WebSockets for this. WebSockets buy you bidirectional traffic you are not using; SSE is simpler and reconnects for free.

## The checkpointer trap in production

Lesson 5.1's checkpointer demos used in-memory or SQLite stores. Both fail the same way in production, just at different scales. The in-memory saver loses everything on restart, which in a container happens constantly. SQLite lives in a file, and a file is not shared between the two replicas of your app that the orchestrator just started. Two requests for the same thread land on different replicas, each sees a different conversation, and your users get an agent with amnesia every other message.

The fix is a checkpointer both replicas can reach: Postgres. The package is `langgraph-checkpoint-postgres`, and the wiring is short:

In [ ]:
# pip install langgraph-checkpoint-postgres==3.2.0  (needs a running Postgres)
from langgraph.checkpoint.postgres import PostgresSaver

DB_URI = "postgresql://agent:secret@db:5432/agentdb"   # from the environment in real code

checkpointer = PostgresSaver.from_conn_string(DB_URI)
checkpointer.setup()          # creates the tables once; safe to call on every boot

graph = builder.compile(checkpointer=checkpointer)

`setup()` is idempotent: run it at startup before the first request. `from_conn_string` takes the same connection string your migrations use. One caution: this saver is synchronous, so it blocks the event loop briefly on each checkpoint write. For the traffic levels this course targets that is fine; the async variant (`langgraph.checkpoint.postgres.aio`) exists for when it is not.

For local development without Docker, SQLite is acceptable, with one correction to the naive version: use the async saver if your endpoints are async.

In [ ]:
# pip install langgraph-checkpoint-sqlite==3.1.2
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

checkpointer = AsyncSqliteSaver.from_conn_string("checkpoints.db")

The rule of thumb: SQLite for your laptop, Postgres for anything with users. And the rule that social media keeps re-learning the hard way: checkpointing every step for every user has no expiry by default, so at scale the checkpoint table becomes a write storm and a storage bill. Keep threads short-lived where you can, and give yourself a retention story. Lesson 6.5 turns that into a GDPR-shaped checklist.

## Configuration and secrets

Every value that differs between your laptop and production comes from the environment, never from the source code. API keys, database URLs, model names, the recursion limit: all of it.

In [ ]:
import os

def require(name: str) -> str:
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(f"missing required environment variable: {name}")
    return value

OPENAI_API_KEY = require("OPENAI_API_KEY")
DB_URI = require("DATABASE_URL")
MODEL = os.environ.get("MODEL", "gpt-4o-mini")       # default is fine; secrets are not

The `require` helper fails at boot, loudly, instead of failing on the first request at 2 AM. Secrets never appear in logs: FastAPI's default exception handler can echo request bodies, so keep keys out of request models entirely. The `.env` file on your laptop is gitignored and never committed; the production values live in the platform's secret store (Render env vars, a `.env` file mounted by compose, AWS Secrets Manager, whichever your host provides).

## Docker: the boring, correct container

The Dockerfile for an agent service is the same as for any Python service, with two agent-specific notes: pin everything with `==` so the container builds the exact stack the lessons were written against, and run as a non-root user.

```dockerfile
FROM python:3.11-slim AS base
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app ./app

RUN useradd --create-home appuser && chown -R appuser /app
USER appuser

EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=5s --retries=3 \
  CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
```

And a compose file for local parity with production: the app plus Postgres, on one network, secrets in a file compose reads but git ignores.

```yaml
services:
  db:
    image: postgres:16
    environment:
      POSTGRES_USER: agent
      POSTGRES_PASSWORD: ${DB_PASSWORD:?set DB_PASSWORD in .env}
      POSTGRES_DB: agentdb
    volumes:
      - pgdata:/var/lib/postgresql/data
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U agent"]
      interval: 10s
      retries: 5

  api:
    build: .
    ports:
      - "8000:8000"
    environment:
      DATABASE_URL: postgresql://agent:${DB_PASSWORD}@db:5432/agentdb
      OPENAI_API_KEY: ${OPENAI_API_KEY:?set OPENAI_API_KEY in .env}
      MODEL: gpt-4o-mini
    env_file:
      - .env
    depends_on:
      db:
        condition: service_healthy

volumes:
  pgdata:
```

The `depends_on` with `service_healthy` is the difference between a compose file that works and one that fails one time in three: without it, the API boots before Postgres accepts connections and `checkpointer.setup()` fails. The `.dockerignore` excludes `.env`, `__pycache__`, and `.git`, so secrets and junk never enter the image.

## The concurrency model, honestly

A common first instinct is to run uvicorn with four workers for throughput. For a streaming agent service, that instinct needs one correction: workers do not share memory, so each worker holds its own compiled graph and, worse, its own view of any in-process checkpointer. With the Postgres checkpointer the state problem disappears, but each worker still holds open SSE connections, and a client reconnecting can land on a different worker mid-conversation. That is survivable: the thread id in the config rehydrates the conversation from Postgres on whichever worker answers.

The production shape is therefore: one uvicorn process per container (or a few), many containers behind a load balancer, one Postgres behind them all. Scale by adding containers, not by adding threads to one container. Model calls are network-bound, so a single async worker handles far more concurrent streams than you expect; the bottleneck is the model provider's rate limit, not your CPU. Give each container a modest memory limit, because lesson 5.4's warning about state bloat applies per process: a runaway graph in one container should kill that container, not the whole host.

One more honest note about free tiers. Platforms like Render's free tier sleep your service after idle time, and the first request after sleep pays a cold start: container boot, dependency import, `checkpointer.setup()`, the first model call. For a demo that is acceptable. For anything with users, it is not, and the answer is a paid tier or a platform without sleep, plus a startup probe so traffic only arrives once `/health` is green.

## What can go wrong

**The stream that is not a stream.** Everything looks right, tokens appear in the server log as they generate, but the browser shows the whole answer at once at the end. The cause is almost always proxy buffering: missing `X-Accel-Buffering: no`, or an nginx config without `proxy_buffering off`, or a CDN in front that buffers by default. Test streaming with `curl -N` (no buffering) directly against the container first; if `curl -N` streams but the browser does not, the buffer is between you and the container.

**Threads that leak across users.** The `thread_id` comes from the client and two users pick `chat-1`. Derive thread ids server-side from the authenticated session. This is a security bug, not a cosmetic one: checkpoint state is conversation history.

**The checkpointer that nobody migrated.** `PostgresSaver` needs its tables. If `setup()` never ran (or ran against a different database than the app uses), the first request fails with a missing-table error. Run `setup()` at boot and log that it ran.

**Secrets in the image.** A `COPY . .` that includes `.env`, or a key baked into an `ENV` line, ships the secret to every registry the image is pushed to. `.dockerignore` the env file, pass secrets at runtime.

**The bill from abandoned streams.** Covered above: without `is_disconnected` checks, closed tabs keep burning model calls. At 2 AM with a runaway loop, this is how the social-media horror stories start. The recursion limit from lesson 2.4 is your backstop; the disconnect check is your front line.

**SQLite in production.** It works until the second replica appears, and then conversations split-brain. The symptom is bizarre: the agent remembers what you said, but only on alternate messages. Move to Postgres before the second replica, not after.

## Check your understanding

**1. Your SSE endpoint streams perfectly when you curl the container directly, but the browser behind your nginx reverse proxy receives the whole answer at once. Name the two fixes, and say which side of the proxy each belongs on.**

On the application side, set the `X-Accel-Buffering: no` response header so nginx knows not to buffer this response. On the proxy side, set `proxy_buffering off` in the nginx location block. The header is a request from the app; the nginx directive is the proxy's own policy. Either one alone usually fixes it, but both is the belt-and-braces version.

**2. You run two API replicas behind a load balancer with the in-memory checkpointer. A user sends message 1 (lands on replica A), then message 2 (lands on replica B). Describe what the user experiences, and name the minimal fix.**

Replica B has no checkpoint for the thread, so the graph starts a fresh conversation: the agent greets the user as a stranger and has no memory of message 1. The user experiences amnesia on alternate messages. The minimal fix is a shared checkpointer both replicas can reach, i.e. Postgres via `langgraph-checkpoint-postgres`, so either replica rehydrates the same thread state.

**3. A user closes the browser tab halfway through a long streamed answer. Your endpoint has no disconnect check. What keeps running, what does it cost, and what is the one-line-shaped fix?**

The async generator keeps iterating `graph.astream` to completion, so the run keeps making model and tool calls for a client that will never read the output; it costs the full remaining API spend of the run. The fix is checking `await request.is_disconnected()` inside the stream loop and breaking out when the client is gone.

**4. Where do the API key and the database URL live in the deployed version of this lesson's app, and why must the boot fail loudly if they are missing?**

They live in environment variables (the platform's secret store or a compose `env_file`), never in source code or the image. Boot must fail loudly, via the `require()` helper, because a missing key discovered on the first real request at 2 AM is an outage, while a missing key discovered at container start is a deploy that never goes out.

**5. Your compose file starts the API and Postgres together, but the API crashes on roughly every third `docker compose up`. The logs show a connection error to the database. What is wrong, and what is the fix?**

The API boots before Postgres is ready to accept connections, so `checkpointer.setup()` fails intermittently. The fix is `depends_on` with `condition: service_healthy` plus a real `healthcheck` on the db service using `pg_isready`, so compose only starts the API after Postgres reports healthy.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.